# Hoja de trabajo 2 
## Task 1
### Integrantes
* Sergio Orellana 221122
* Rodrigo Mansilla 22611
* Ricardo Chuy 221007

## Pregunta 1

Elijan un sistema real que pueda modelarse como un k-armed bandit. Su modelado debe incluir:

### 1.a. Definición formal de los k brazos

¿Qué representa cada acción y por qué el conjunto de acciones es finito y discreto en este dominio?

- _Respuesta:_ Ya que no se tienen estados en este tipo de problemas, pensamos en modelar el sistema de un e-commerce que trata de vender un producto en específico. Digamos que hay un producto en la tienda que se quiere vender, pero no se sabe cual podría ser el precio más apropiado. Lo que se busca es poder decidir a qué precio mostrárselo a cada usuario que visita la página. Los k brazos son los k precios posibles que la tienda puede mostrar. Por ejemplo k = 5 con precios: $49, $59, $69, $79, $89. Cada vez que un usuario llega a la página, el sistema elige uno de estos precios y lo muestra, esa es una acción. El conjunto de las acciones es finito, ya que la tienda define un catálogo de precios posibles, no es que se pueda mostrar cualquier número real. Es discreto porque cada precio es un opción separada y no son infinitas opciones, no hay precio intermedio entre $49 y $59 en este modelo. 

### 1.b. Diseño justificado de la función de recompensa

¿Qué se mide, cómo se mide, y qué supuestos distribucionales son razonables para ese dominio? Si la recompensa no es gaussiana, arguméntenlo.

_Respuesta:_ 


Lo que se mide es el ingreso generado cuando un usuario ve el precio. $R_t$ es la recompensa recibida en el paso $t$, formalmente:

$$R_t = \begin{cases} p_a & \text{si el usuario compra} \\ 0 & \text{si no compra} \end{cases}$$

donde $p_a$ es el precio asociado al brazo $a$ elegido en el paso $t$.

La recompensa no es gaussiana porque el resultado es binario: el usuario compra o no. La probabilidad de cada caso es:

$$P(R_t = p_a) = \theta_a \qquad P(R_t = 0) = 1 - \theta_a$$

donde $\theta_a$ es la probabilidad de compra bajo el precio $p_a$. A mayor precio, menor $\theta_a$, pero mayor ganancia si la compra ocurre.

### 1.c. Análisis de estacionariedad

¿Los valores verdaderos $q_*(a)$ cambian con el tiempo en este sistema? ¿En qué escala temporal? ¿Qué implicaciones tiene eso para el algoritmo de actualización?

_Respuesta:_


Los valores verdaderos $q_*(a)$ sí cambian con el tiempo, en una escala de días o semanas. La probabilidad de que alguien compre a $89 varía si hay una temporada de descuentos, si un competidor baja sus precios, o si hay una crisis de alguna forma que afecte el mercado o bien al mismo negocio.

Usar promedio muestral $\frac{1}{n}$ es inadecuado aquí porque le da el mismo peso a una observación de hace tres meses que a una de ayer. Lo correcto es usar paso constante $\alpha$ en la regla de actualización:

$$Q_{n+1} = Q_n + \alpha(R_n - Q_n)$$

esto garantiza que las observaciones recientes pesen más, permitiendo que el sistema se adapte a los cambios del mercado y al contexto en el cual se están intentando hacer las ventas. Dependiendo del tiempo puede que una decisión actual tenga influencia por el momento.

### 1.d. Identificación de restricciones de exploración

¿Hay costos económicos, de seguridad o regulatorios que limiten cuánto puede explorar el sistema?

_Respuesta:_

Se pueden destacar 3 restricciones reales importantes.

- La primera es económica, ya que mostrarle a un usuario un precio alto cuando probablemente no va a comprar tiene un costo directo, ese usuario se va sin comprar y posiblemente no regresa. Explorar precios extremos tiene un costo de oportunidad real. Un precio alto aunque genere más ganacias puede que no genere tantas recompensas necesariamente.

- La segunda también es económica, de igual forma relacionada al precio. Si el negocio prueba con precios demasiado bajos para garantizar más compras, las personas pueden sentir desconfianza del producto y la marca, sin mencioanr que también pueden ocasionar perdidas demasiado grandes si se experimentan con presios muy reducidos por largos plazos de tiempo.

- La tercera es en cierta forma de reputación si el sistema muestra precios muy distintos a usuarios diferentes en el mismo momento y estos lo comparan, puede generar desconfianza en la marca. Esto limita cuánto puede explorar el sistema en simultáneo.

## Pregunta 2

Con base en el análisis anterior, propongan y justifiquen:

### 2.a. Estrategia de selección de acción

La estrategia de selección de acción más apropiada para su dominio, con el valor concreto del hiperparámetro relevante ($\varepsilon$ o $c$) y la justificación de ese valor.

_Respuesta:_

La estrategia más apropiada para este dominio es UCB. Greedy puro queda descartado porque nunca explora, lo cual es peligroso en un mercado cambiante. ε-greedy explora aleatoriamente, lo que podría mostrar precios extremos sin ninguna razón inteligente, violando las restricciones de reputación y costo de oportunidad mencionadas anteriormente.

UCB explora de forma un poco más inteligente priorizando acciones con alta incertidumbre, lo cual respeta mejor las restricciones del dominio: solo explora un precio poco visitado cuando hay razón genuina para hacerlo.

$$A_t = \arg\max_a \left[ Q_t(a) + c\sqrt{\frac{\ln t}{N_t(a)}} \right]$$

donde $N_t(a)$ es el número de veces que se mostró el precio $a$ hasta el paso $t$, y $c$ controla qué tan agresivo es el sistema al explorar. Se propone $c = 0.5$ ya que el costo de exploración es alto en este dominio, por lo que se prefiere un nivel de confianza conservador que no fuerce exploración innecesaria.

### 2.b. Regla de actualización de estimaciones

La regla de actualización de estimaciones (paso variable o constante) con justificación ligada al análisis de estacionariedad.

_Respuesta:_

Elegimos una regla de actualización con **paso constante** $\alpha = 0.20$, debido a que el sistema es no estacionario y las probabilidades de compra pueden cambiar con el tiempo. La actualización para el precio seleccionado se realiza mediante:

$$Q_{t+1}(a) = Q_t(a) + \alpha \left[R_t - Q_t(a)\right]$$

En esta fórmula, $Q_t(a)$ representa el ingreso esperado estimado para el precio $a$, mientras que $R_t$ corresponde al ingreso observado después de mostrar ese precio. Por lo tanto, $\alpha = 0.20$ indica que cada nueva observación corrige el 20 % del error existente entre la recompensa observada y la estimación anterior.

No utilizamos un paso variable de la forma $\frac{1}{N_t(a)}$, porque este disminuye conforme aumenta el número de observaciones y les da demasiado peso a los datos históricos. En cambio, el paso constante conserva la capacidad de adaptación ante temporadas de descuentos, cambios en la competencia o variaciones en la disposición de compra de los usuarios. Además, mantenemos $N_t(a)$ para calcular el término de exploración de UCB, pero actualizamos $Q_t(a)$ mediante el paso constante.

### 2.c. Ejemplo numérico

Un ejemplo numérico con al menos seis observaciones que ilustre cómo evoluciona $Q_t(a)$ para dos acciones distintas bajo la regla elegida, mostrando cómo la estimación converge o se adapta.

_Respuesta:_

Para ilustrar la actualización, comparamos dos acciones: mostrar el producto a \$59 y mostrarlo a \$79. Inicializamos ambas estimaciones en cero y utilizamos $\alpha = 0.20$:

$$Q_0(59) = Q_0(79) = 0$$

La recompensa corresponde al precio mostrado cuando el usuario compra y a cero cuando no compra. Por ejemplo, si se muestra el precio de $59 y el usuario compra, entonces $R_t = 59$; en cambio, si no compra, entonces $R_t = 0$.

| Observación $t$ | Precio seleccionado | Recompensa $R_t$ | $Q_t(59)$ | $Q_t(79)$ |
|---:|---:|---:|---:|---:|
| 1 | $59 | 59 | 11.8000 | 0.0000 |
| 2 | $79 | 0 | 11.8000 | 0.0000 |
| 3 | $59 | 59 | 21.2400 | 0.0000 |
| 4 | $79 | 79 | 21.2400 | 15.8000 |
| 5 | $59 | 0 | 16.9920 | 15.8000 |
| 6 | $79 | 0 | 16.9920 | 12.6400 |
| 7 | $59 | 59 | 25.3936 | 12.6400 |
| 8 | $79 | 79 | 25.3936 | 25.9120 |
| 9 | $59 | 0 | 20.3149 | 25.9120 |
| 10 | $79 | 0 | 20.3149 | 20.7296 |
| 11 | $59 | 59 | 28.0519 | 20.7296 |
| 12 | $79 | 0 | 28.0519 | 16.5837 |

En la primera observación, la estimación del precio de $59 se actualiza de la siguiente manera:

$$Q_1(59) = 0 + 0.20(59 - 0) = 11.80$$

Posteriormente, en la quinta observación, el usuario no compra al precio de $59; por consiguiente, la estimación disminuye:

$$Q_5(59) = 21.24 + 0.20(0 - 21.24) = 16.992$$

Después de las doce observaciones, obtenemos $Q_t(59) \approx 28.05$ y $Q_t(79) \approx 16.58$. Por esta razón, el precio de $59 presenta un ingreso esperado reciente mayor. Sin embargo, la estimación del precio de $79 aumentó cuando produjo una compra y volvió a disminuir después de varios intentos sin venta.